In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Basic libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb

# Load data
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

print(train.shape, test.shape)
train.head()


In [ ]:
# Summary of dataset
train.info()
train.describe()

# Check missing values
missing = train.isnull().sum().sort_values(ascending=False)
missing[missing > 0].head(20)


In [ ]:
# SalePrice distribution
plt.figure(figsize=(8,6))
sns.histplot(train["SalePrice"], kde=True, bins=30, color="blue")
plt.title("Distribution of Sale Prices")
plt.show()

# Correlation heatmap (numeric features only)
plt.figure(figsize=(12,10))
corr = train.corr(numeric_only=True)   # <--- IMPORTANT
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

# Scatter plot (GrLivArea vs SalePrice)
plt.figure(figsize=(8,6))
sns.scatterplot(x=train["GrLivArea"], y=train["SalePrice"])
plt.title("GrLivArea vs SalePrice")
plt.show()

# Boxplot (OverallQual vs SalePrice)
plt.figure(figsize=(8,6))
sns.boxplot(x=train["OverallQual"], y=train["SalePrice"])
plt.title("Sale Price by Overall Quality")
plt.show()


In [ ]:
# Top correlations with SalePrice
corr_matrix = train.corr(numeric_only=True)
top_corr = corr_matrix["SalePrice"].sort_values(ascending=False).head(10)

plt.figure(figsize=(8,6))
sns.barplot(x=top_corr.values, y=top_corr.index, palette="viridis")
plt.title("Top 10 Features Correlated with SalePrice")
plt.show()


In [ ]:
# --- Handle Missing Values ---

# Fill numeric NaNs with median
for col in train.select_dtypes(include=[np.number]).columns:
    if col != "SalePrice":  # don't touch the target column
        train[col] = train[col].fillna(train[col].median())
        test[col] = test[col].fillna(test[col].median())

# Fill categorical NaNs with mode (most frequent value)
for col in train.select_dtypes(include=['object']).columns:
    train[col] = train[col].fillna(train[col].mode()[0])
    test[col] = test[col].fillna(test[col].mode()[0])

print("Missing values in train:", train.isnull().sum().sum())
print("Missing values in test:", test.isnull().sum().sum())


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Combine train (without SalePrice) and test for consistent encoding
all_data = pd.concat([train.drop("SalePrice", axis=1), test], axis=0)

# Encode categorical columns
for col in all_data.select_dtypes(include="object").columns:
    lbl = LabelEncoder()
    all_data[col] = lbl.fit_transform(all_data[col].astype(str))

# Split back into train and test
X = all_data.iloc[:train.shape[0], :]
X_test = all_data.iloc[train.shape[0]:, :]
y = train["SalePrice"]

print("Train shape:", X.shape, "Test shape:", X_test.shape)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, "Validation:", X_val.shape)


In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import numpy as np

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42)
}

rmse_scores = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    rmse_scores[name] = rmse
    print(f"{name}: RMSE = {rmse:.2f}")


In [ ]:
import pandas as pd

rmse_table = pd.DataFrame.from_dict(rmse_scores, orient="index", columns=["RMSE"])
rmse_table.sort_values(by="RMSE")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Using Random Forest (replace with your best model)
best_model = RandomForestRegressor(n_estimators=200, random_state=42)
best_model.fit(X, y)

# Get feature importances
importances = best_model.feature_importances_
features = X.columns

# Put into DataFrame
feat_imp = pd.DataFrame({
    "Feature": features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False).head(10)

# Plot Top 10 Features
plt.figure(figsize=(8,6))
sns.barplot(x="Importance", y="Feature", data=feat_imp, palette="viridis")
plt.title("Top 10 Feature Importances")
plt.show()


In [ ]:
# Train the best model on full training data
final_model = RandomForestRegressor(n_estimators=200, random_state=42)
final_model.fit(X, y)

# Predict on test set
predictions = final_model.predict(X_test)

# Create submission DataFrame
submission = pd.DataFrame({
    "Id": test["Id"],   # Id column from test.csv
    "SalePrice": predictions
})

# Save to CSV
submission.to_csv("submission.csv", index=False)

print("Submission file created successfully!")


In this lab I joined the Kaggle House Prices competition and built models to predict sale prices.
I cleaned the data, handled missing values, and encoded text columns. I made some graphs to explore the data and checked which features are most important. Then I trained different models like Linear Regression, Decision Tree, Random Forest, and Gradient Boosting. I compared them and chose the best model with the lowest RMSE. Finally, I made predictions on the test set and created a Kaggle submission file.
I learned how to prepare data, compare models, and understand which features affect house prices. Next time I can try tuning the models, testing XGBoost or LightGBM, and doing more feature engineering to get better accuracy.
